# Station Proximity Statistics 

Computes three descriptive statistics to explain the **FM/LM asymmetry** observed in the gravity-based accessibility model, using **exact network walk distances** (consistent with `compute_gravity.ipynb`):

1. **% of population** within 8-min walk of a bike-sharing station
2. **% of POIs** within 8-min walk of a bike-sharing station (by category)
3. **% of PT stops** within 8-min walk of a bike-sharing station

---

## Required Files


| File | Description |
|---|---|
| `lux_walk.graphml` | Walk network |
| `origins.csv` | Grid cell origins with population (`<30` cells set to 15, as in `compute_theil.ipynb`) |
| `pois.csv` | Points of interest with category |
| `stops_only_luxemburg.csv` | PT stop coordinates |
| `bike_stations_merged.csv` | Bike-sharing station coordinates |


In [1]:
import math
import os
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
import geopandas as gpd
from tqdm.notebook import tqdm

print("All imports OK.")

All imports OK.


## 0. Configuration

In [1]:
# ── Paths ──────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"
CACHE_DIR    = PROJECT_ROOT / "cache"

WALK_GRAPH   = str(CACHE_DIR / "lux_walk.graphml")
ORIGINS_CSV  = str(DATA_DIR / "origins.csv")
POIS_CSV     = str(DATA_DIR / "pois.csv")
GTFS_STOPS   = str(DATA_DIR / "stops_only_luxemburg.csv")
STATIONS_CSV = str(DATA_DIR / "bike_stations_merged.csv")

# ── CRS ────────────────────────────────────────────────────────
CRS_PROJ = "EPSG:2169"   # Luxembourg national grid
CRS_GEO  = "EPSG:4326"   # WGS84

# ── Walk threshold (mirrors compute_gravity.ipynb) ─────────────
WALK_STATION = 8.0        # minutes
INF          = 1e9

# ── Suppressed population cells (mirrors compute_theil.ipynb) ──
SUPPRESSED_MIDPOINT = 15  # '<30' cells → midpoint of 15 residents

# ── Tobler walk parameters (mirrors compute_gravity.ipynb) ─────
STAIR_SPEED_M_PER_MIN = 42.0

def tobler_speed_kmh(grade_dec: float) -> float:
    return 6.0 * math.exp(-3.5 * abs(grade_dec + 0.05))

print(f"Walk station threshold : {WALK_STATION} min")
print(f"Data directory         : {DATA_DIR}")


NameError: name 'Path' is not defined

## 1. Load & Prepare Walk Graph

In [13]:
print("Loading walk graph (this may take a moment)...")
G_walk = ox.load_graphml(WALK_GRAPH)

print("Assigning slope-adjusted walk_time_min to edges...")
bad_edges = []
for u, v, k, data in G_walk.edges(keys=True, data=True):
    length_m = float(data.get("length", 0.0) or 0.0)
    grade    = float(data.get("grade",  0.0) or 0.0)
    highway  = data.get("highway", "")
    is_steps = ("steps" in highway) if isinstance(highway, list) else (highway == "steps")

    if length_m <= 0:
        data["walk_time_min"] = None
        bad_edges.append((u, v, k))
        continue

    if is_steps:
        data["walk_time_min"] = length_m / STAIR_SPEED_M_PER_MIN
    else:
        speed_kmh = tobler_speed_kmh(grade)
        data["walk_time_min"] = ((length_m / 1000.0) / speed_kmh) * 60.0

G_walk.remove_edges_from(bad_edges)
print(f"  Dropped {len(bad_edges)} edges with invalid length.")
print(f"  Walk graph: {len(G_walk.nodes):,} nodes, {len(G_walk.edges):,} edges")


Loading walk graph (this may take a moment)...
Assigning slope-adjusted walk_time_min to edges...
  Dropped 0 edges with invalid length.
  Walk graph: 99,289 nodes, 254,087 edges


## 2. Load & Reproject Data

In [4]:
print("Loading data files...")

origins      = pd.read_csv(ORIGINS_CSV)
pois         = pd.read_csv(POIS_CSV)
stops_raw    = pd.read_csv(GTFS_STOPS, dtype={"stop_id": str})
stations_raw = pd.read_csv(STATIONS_CSV)

# Parse Pop_grids with the same rule as compute_theil.ipynb:
# exact numbers kept as-is; '<30' (suppressed) cells → SUPPRESSED_MIDPOINT.
def parse_pop_value(x, midpoint=SUPPRESSED_MIDPOINT):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    try:
        return float(s)
    except ValueError:
        pass
    if re.match(r"^\s*<=?\s*30\s*$", s):
        return float(midpoint)
    m = re.search(r"(\d+(\.\d+)?)", s)
    if m:
        return float(m.group(1))
    return np.nan

origins["is_suppressed"] = origins["Pop_grids"].apply(
    lambda x: bool(re.match(r"^\s*<=?\s*30\s*$", str(x).strip())) if isinstance(x, str) else False
)
origins["pop"] = origins["Pop_grids"].apply(parse_pop_value).fillna(0)

print(f"  Suppressed cells (<30) : {origins['is_suppressed'].sum():,}  (set to {SUPPRESSED_MIDPOINT} residents each)")
print(f"  Total population       : {origins['pop'].sum():,.0f}")

# Reproject stops and stations from WGS84 → EPSG:2169
stops = gpd.GeoDataFrame(
    stops_raw.copy(),
    geometry=gpd.points_from_xy(stops_raw["lon"], stops_raw["lat"]),
    crs=CRS_GEO,
).to_crs(CRS_PROJ)
stops["x"] = stops.geometry.x
stops["y"] = stops.geometry.y

stations = gpd.GeoDataFrame(
    stations_raw,
    geometry=gpd.points_from_xy(stations_raw["longitude"], stations_raw["latitude"]),
    crs=CRS_GEO,
).to_crs(CRS_PROJ)
stations["x"] = stations.geometry.x
stations["y"] = stations.geometry.y

print(f"  Origins  : {len(origins):,}")
print(f"  POIs     : {len(pois):,}")
print(f"  PT stops : {len(stops):,}")
print(f"  Stations : {len(stations):,}")


Loading data files...
  Suppressed cells (<30) : 567  (set to 15 residents each)
  Total population       : 646,170
  Origins  : 2,795
  POIs     : 13,497
  PT stops : 2,522
  Stations : 290


## 3. Map Entities to Nearest Walk Graph Nodes

In [5]:
print("Mapping entities to nearest walk graph nodes...")

origins["walk_node"]  = ox.nearest_nodes(G_walk, origins["x"].values,  origins["y"].values)
pois["walk_node"]     = ox.nearest_nodes(G_walk, pois["x"].values,     pois["y"].values)
stops["walk_node"]    = ox.nearest_nodes(G_walk, stops["x"].values,    stops["y"].values)
stations["walk_node"] = ox.nearest_nodes(G_walk, stations["x"].values, stations["y"].values)

print("  Done.")


Mapping entities to nearest walk graph nodes...
  Done.


## 4. Build Reachability Set from All Stations

Run single-source Dijkstra from each bike station node (cutoff = 8 min).  
Collect the **union** of all reachable walk-graph nodes across all stations.

In [7]:
print(f"Running Dijkstra from each station (cutoff = {WALK_STATION} min)...")
print("This runs 290 Dijkstra searches — expect ~2–5 minutes.")

reachable_nodes = set()

G_rev = G_walk.reverse(copy=False)




for row in tqdm(stations.itertuples(), total=len(stations), desc="Stations"):
    try:
        lengths = nx.single_source_dijkstra_path_length(
            G_rev, row.walk_node, cutoff=WALK_STATION, weight="walk_time_min"
            )
        reachable_nodes.update(lengths.keys())
    except nx.NodeNotFound:
        continue

print(f"\n  Unique walk-graph nodes reachable from at least one station: {len(reachable_nodes):,}")


Running Dijkstra from each station (cutoff = 8.0 min)...
This runs 290 Dijkstra searches — expect ~2–5 minutes.


Stations:   0%|          | 0/290 [00:00<?, ?it/s]


  Unique walk-graph nodes reachable from at least one station: 33,322


## 5. Stat 1 — Population within 8-min Walk of a Bike Station

In [8]:
origins["near_station"] = origins["walk_node"].isin(reachable_nodes)

total_pop = origins["pop"].sum()
pop_near  = origins.loc[origins["near_station"], "pop"].sum()
pct_pop   = pop_near / total_pop * 100
pct_cells = origins["near_station"].mean() * 100

print("STAT 1 — Population within 8-min walk of a bike station")
print(f"  Total population         : {int(total_pop):>10,}")
print(f"  Population near station  : {int(pop_near):>10,}  ({pct_pop:.1f}%)")
print(f"  Grid cells near station  : {origins['near_station'].sum():>10,} / {len(origins):,}  ({pct_cells:.1f}%)")

# Suppressed cells near a station (for transparency)
supp      = origins["is_suppressed"]
supp_near = (supp & origins["near_station"]).sum()
print(f"  Suppressed cells near    : {supp_near:>10,} / {supp.sum():,}  ({supp_near / supp.sum() * 100:.1f}%)")


STAT 1 — Population within 8-min walk of a bike station
  Total population         :    646,170
  Population near station  :    274,721  (42.5%)
  Grid cells near station  :        125 / 2,795  (4.5%)
  Suppressed cells near    :          7 / 567  (1.2%)


## 6. Stat 2 — POIs within 8-min Walk of a Bike Station

In [9]:
pois["near_station"] = pois["walk_node"].isin(reachable_nodes)

total_pois = len(pois)
pois_near  = pois["near_station"].sum()
pct_pois   = pois_near / total_pois * 100

print("STAT 2 — POIs within 8-min walk of a bike station")
print(f"  Total POIs               : {total_pois:>10,}")
print(f"  POIs near station        : {pois_near:>10,}  ({pct_pois:.1f}%)")
print(f"\n  {'Category':<25} {'Near':>6} {'Total':>6} {'%':>6}")
print(f"  {'-'*25} {'-'*6} {'-'*6} {'-'*6}")
for cat, grp in pois.groupby("category"):
    n      = len(grp)
    n_near = grp["near_station"].sum()
    pct    = n_near / n * 100
    print(f"  {cat:<25} {n_near:>6,} {n:>6,} {pct:>6.1f}%")


STAT 2 — POIs within 8-min walk of a bike station
  Total POIs               :     13,497
  POIs near station        :      6,088  (45.1%)

  Category                    Near  Total      %
  ------------------------- ------ ------ ------
  Bars and restaurants       1,299  2,292   56.7%
  Basic Healthcare             152    313   48.6%
  Commercial                 1,623  2,623   61.9%
  Education                    250    525   47.6%
  Entertainment              1,143  3,850   29.7%
  Food                         261    595   43.9%
  Jobs                       1,229  2,918   42.1%
  Parks                        131    381   34.4%


## 7. Stat 3 — PT Stops within 8-min Walk of a Bike Station

In [10]:
stops["near_station"] = stops["walk_node"].isin(reachable_nodes)

total_stops = len(stops)
stops_near  = stops["near_station"].sum()
pct_stops   = stops_near / total_stops * 100

print("STAT 3 — PT stops within 8-min walk of a bike station")
print(f"  Total PT stops           : {total_stops:>10,}")
print(f"  PT stops near station    : {stops_near:>10,}  ({pct_stops:.1f}%)")


STAT 3 — PT stops within 8-min walk of a bike station
  Total PT stops           :      2,522
  PT stops near station    :        762  (30.2%)


## 8. Summary

In [11]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  % population near bike station  : {pct_pop:.1f}%")
print(f"  % POIs near bike station        : {pct_pois:.1f}%")
print(f"  % PT stops near bike station    : {pct_stops:.1f}%")
print("=" * 60)

SUMMARY
  % population near bike station  : 42.5%
  % POIs near bike station        : 45.1%
  % PT stops near bike station    : 30.2%

Interpretation:
  Bike stations are accessible from 45% of POIs and 30% of PT stops,
  but only 4% of grid cells (holding 43% of the population) live within
  an 8-minute walk of at least one bike-sharing station. This spatial mismatch
  explains why e-bike egress (Scenario C) benefits far more grid cells than
  e-bike access (Scenario B).

